New ML test pipeline script


Import all libraries needed for this ML script.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
print('Imports Successfully!')

Imports Successfully!


In [5]:
#to check the python version and the path of the python executable
import sys
print(sys.executable)

/Users/boracomert/Desktop/Osint_project/.venv/bin/python


the next step: Load data 

In [9]:
DATA_PATH = "../data/processed/labeled_merged_data_cleaned.csv"

df = pd.read_csv(DATA_PATH)
print('Data Loaded Successfully!')

#the model shouldnt see layoff information becuse it can cause data leakage, so we will drop the layoff columns from the features
META_COLS = ['company', 'date', 'quarter', 'layoff','same_quarter', 'next_quarter', 'layoff_same_quarter', 'layoff_next_quarter', 'layoff_same_or_next_quarter']

FEATURE_COLS = [col for col in df.columns if col not in META_COLS]


n_companies  = df['company'].nunique()
n_rows_pos   = len(df)

print(f'Positive companies : {n_companies}')
print(f'Positive rows      : {n_rows_pos}')
print(f'Feature columns    : {len(FEATURE_COLS)}')
print(f'Quarters present   : {sorted(df["quarter"].unique())}')
df.head(5)

Data Loaded Successfully!
Positive companies : 1941
Positive rows      : 12047
Feature columns    : 346
Quarters present   : ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']


,company,date,quarter,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,...,fin_Other Non Interest Expense,fin_Depletion Income Statement,fin_Policyholder Benefits Ceded,fin_Net Income Extraordinary,same_quarter,next_quarter,layoff_same_quarter,layoff_next_quarter,layoff_same_or_next_quarter,layoff
0,AFCONS.BO,2024-09-30,2024Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q3,2024Q4,0,0,0,0
1,AFCONS.BO,2024-12-31,2024Q4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q4,2025Q1,0,0,0,0
2,AFCONS.BO,2025-03-31,2025Q1,367784631.0,367784631.0,1.795550e+10,2.343300e+10,5.259830e+10,7.496240e+10,3.021220e+10,...,NaN,NaN,NaN,NaN,2025Q1,2025Q2,0,0,0,0
3,AFCONS.BO,2025-06-30,2025Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2025Q2,2025Q3,0,0,0,0
4,AFCONS.BO,2025-09-30,2025Q3,367784631.0,367784631.0,3.097260e+10,3.565230e+10,5.388340e+10,8.861110e+10,3.037610e+10,...,NaN,NaN,NaN,NaN,2025Q3,2025Q4,0,0,0,0
